# Segments

The model ranks individual customers. This notebook groups them into segments a retention team can act on, sizes the money at stake in each, and designs a proper experiment for the biggest one.

In [1]:
import sqlite3
import pandas as pd

con = sqlite3.connect("../churn.db")
df = pd.read_sql("select * from customers", con)

# model scores from notebook 02
scores = pd.read_csv("../exports/risk_scores.csv")[["customer_id", "churn_prob"]]
df = df.merge(scores, on="customer_id")
len(df)

10000

Candidate segments straight from the EDA and the importance chart: the inactive-older cliff, the Germany problem, the 3-4 product trap, and the under-engaged single-product pool. "Balance at risk" weights each customer's balance by their churn probability - a rough euro figure for what walks out the door if nothing changes. (Caveat: 80% of these scores are in-sample, so treat them as directional.)

In [2]:
segs = {
    "inactive 50+, funded": (df.is_active == 0) & (df.age >= 50) & (df.balance > 0),
    "germany, balance 100k+": (df.geography == "Germany") & (df.balance >= 100000),
    "3-4 products": df.num_products >= 3,
    "1 product, funded, inactive": (df.num_products == 1) & (df.balance > 0) & (df.is_active == 0),
}

rows = []
for name, m in segs.items():
    s = df[m]
    rows.append({
        "segment": name,
        "customers": len(s),
        "churn_pct": round(100 * s.exited.mean(), 1),
        "avg_balance": round(s.balance.mean()),
        "avg_churn_prob": round(s.churn_prob.mean(), 2),
        "balance_at_risk_m": round((s.balance * s.churn_prob).sum() / 1e6, 1),
    })
pd.DataFrame(rows)

,segment,customers,churn_pct,avg_balance,avg_churn_prob,balance_at_risk_m
0,"inactive 50+, funded",356,84.0,119913,0.94,40.2
1,"germany, balance 100k+",1966,35.9,129771,0.55,137.0
2,3-4 products,326,85.9,78822,0.92,24.9
3,"1 product, funded, inactive",2048,34.2,120875,0.56,138.5


In [3]:
# how much these overlap
a, b, d = segs["inactive 50+, funded"], segs["germany, balance 100k+"], segs["1 product, funded, inactive"]
print("inactive 50+ also in germany 100k+:", (a & b).sum())
print("inactive 50+ also 1-product:", (a & d).sum())

inactive 50+ also in germany 100k+: 152
inactive 50+ also 1-product: 249


Reading the table:

- **Inactive 50+, funded** - 356 customers, **84% churn**, avg balance 120k, ~40M at risk. The densest risk pocket in the book and exactly the shape a retention offer can reach. This is the experiment segment.
- **Germany, 100k+ balances** - 1,966 customers at 36% churn, 137M at risk. Too big and too structural for a call list; even active Germans churn above base rate, so this goes to a product and pricing review, not to outreach.
- **3-4 products** - 326 customers at **86% churn**. Near-certain loss. Nobody should A/B test this; someone should work out what these bundles actually cost the customer.
- **1 product, funded, inactive** - 2,048 customers at 34%, the biggest pool (138M at risk). Cross-sell to a second product is the obvious play, but the 2-product sweet spot is correlational, so it needs its own experiment. Second in the queue.

Overlaps are real (152 of the German group are also inactive 50+), so any rollout needs dedupe rules - experiment segment wins, everyone else stays eligible for their own track.

## Proposed retention experiment

**Hypothesis.** A personal outreach call plus a 12-month fee waiver reduces 6-month churn among inactive, funded 50+ customers by at least 15 percentage points.

**Design.** Randomize the 356 segment customers 1:1 - treatment gets a banker call within two weeks plus the offer (~30 EUR cost per contact), control gets nothing beyond business as usual.

**Primary metric.** 6-month account closure rate, treatment vs control.

**Secondary.** Reactivation (any transaction within 90 days), balance retention at 6 months.

**Guardrails.**
- cost per incrementally retained customer must stay under the 250 EUR/year margin - if the offer mostly goes to people who would have stayed anyway, that number blows up
- complaint / opt-out rate on the outbound calls
- no peeking for efficacy before the 6-month readout; early stop only on guardrail breach

**Assumptions** (stated, since the dataset has no timestamps): the observed 84% churn is roughly a 12-month figure. Constant hazard converts that to a 6-month baseline of ~60%, which the power calc below uses.

In [4]:
from math import ceil, sqrt
from scipy.stats import norm

p12 = 0.84                # observed churn in the segment, assumed annual
p6 = 1 - sqrt(1 - p12)    # constant hazard -> 6-month rate

def n_per_arm(p1, p2, alpha=0.05, power=0.8):
    za, zb = norm.ppf(1 - alpha / 2), norm.ppf(power)
    pbar = (p1 + p2) / 2
    num = (za * sqrt(2 * pbar * (1 - pbar)) + zb * sqrt(p1 * (1 - p1) + p2 * (1 - p2))) ** 2
    return ceil(num / (p1 - p2) ** 2)

print(f"6-month baseline: {p6:.0%}")
print(f"15pp reduction ({p6:.0%} -> {p6 - 0.15:.0%}): {n_per_arm(p6, p6 - 0.15)} per arm")
print(f"10pp reduction ({p6:.0%} -> {p6 - 0.10:.0%}): {n_per_arm(p6, p6 - 0.10)} per arm")

6-month baseline: 60%
15pp reduction (60% -> 45%): 173 per arm
10pp reduction (60% -> 50%): 388 per arm


**Sample size.** Detecting a 15pp reduction (60% to 45%) at alpha 0.05 / 80% power needs ~173 per arm - 346 customers, and the segment has 356. It fits, barely. A 10pp effect would need ~390 per arm, which this 10k book cannot supply; at a real bank's scale the same segment is thousands of customers and that constraint disappears. So: this experiment is powered for a 15pp effect, and anything smaller reads as inconclusive, not as absence of effect.

**Duration.** 2-week rollout of calls, then the 6-month observation window. Readout at month 7.

**Decision rule.** Two-sided two-proportion z-test at the readout. Ship the program to the full segment (and start the 1-product cross-sell experiment next) if the reduction is significant at 0.05 **and** cost per incrementally retained customer is under 250 EUR. The economics have slack: 178 calls cost ~5.3k EUR; a 15pp effect is ~27 extra customers retained, worth ~6.7k EUR/year in margin plus ~3M EUR in balances that stay.